# AgenticLens Beginner Demo: RAG Workflow

This notebook is for a first-time AgenticLens user. It shows a simple RAG flow step by step.

You will track:

1. User prompt
2. Retrieval
3. LLM answer generation
4. Token usage
5. Latency
6. Cost
7. AgenticLens report and analysis

Default mode is mock mode, so it runs without an API key.

In [ ]:
# Optional. Run only if imports fail.
# !pip install agenticlens openai pandas matplotlib

## 1. Imports

In [ ]:
import os
import time
import pandas as pd
import matplotlib.pyplot as plt
from getpass import getpass

from agenticlens import profile, step
from agenticlens.exporters import JSONExporter

## 2. Choose Mock Mode or Real OpenAI Mode

Start with `USE_REAL_OPENAI = False`. Change it to `True` only when you want real API calls.

In [ ]:
USE_REAL_OPENAI = False
MODEL_NAME = "gpt-4o-mini"

if USE_REAL_OPENAI:
    from openai import OpenAI
    if not os.getenv("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY: ")
    client = OpenAI()
else:
    client = None

## 3. Create a Tiny Knowledge Base

In real RAG, these documents can come from PDFs, websites, databases, or vector stores.

In [ ]:
DOCS = [
    "Refund policy: Customers can request a refund within 30 days of delivery.",
    "Refund policy: Items must be unused and in original packaging.",
    "Refunds are processed to the original payment method.",
    "Refunds may take 5 to 10 business days.",
    "Shipping policy: Standard shipping takes 5 to 7 business days.",
    "Express shipping takes 2 to 3 business days.",
    "Order tracking: Customers can track orders using their order ID.",
]

## 4. User Question

In [ ]:
user_question = "Can I get a refund and how long will it take?"
print(user_question)

## 5. Retriever

This is a simple keyword retriever. It lets users understand the flow before moving to embeddings/vector search.

In [ ]:
def clean_words(text: str):
    return set(text.lower().replace("?", "").replace(".", "").replace(",", "").split())

def retrieve_context(query: str, top_k: int = 5):
    query_words = clean_words(query)
    scored_docs = []
    for doc in DOCS:
        score = len(query_words.intersection(clean_words(doc)))
        scored_docs.append((score, doc))
    scored_docs.sort(reverse=True, key=lambda item: item[0])
    return [doc for score, doc in scored_docs[:top_k] if score > 0]

## 6. LLM Response

Mock mode creates an OpenAI-like response with token usage, so AgenticLens can record it.

In [ ]:
class MockUsage:
    def __init__(self, prompt_tokens: int, completion_tokens: int):
        self.prompt_tokens = prompt_tokens
        self.completion_tokens = completion_tokens

class MockMessage:
    def __init__(self, content: str):
        self.content = content

class MockChoice:
    def __init__(self, content: str):
        self.message = MockMessage(content)

class MockResponse:
    def __init__(self, content: str, prompt_tokens: int, completion_tokens: int):
        self.usage = MockUsage(prompt_tokens, completion_tokens)
        self.choices = [MockChoice(content)]


def call_llm(question: str, chunks: list[str]):
    context = "\n".join(f"- {chunk}" for chunk in chunks)

    if USE_REAL_OPENAI:
        messages = [
            {"role": "system", "content": "You are a helpful support assistant. Use only the provided context."},
            {"role": "user", "content": "Question:\n" + question + "\n\nContext:\n" + context + "\n\nAnswer clearly."},
        ]
        return client.chat.completions.create(model=MODEL_NAME, messages=messages, temperature=0)

    answer = "Yes. You can request a refund within 30 days of delivery if the item is unused and in original packaging. Refunds may take 5 to 10 business days."
    return MockResponse(answer, prompt_tokens=180, completion_tokens=55)

## 7. Start AgenticLens Profiling

This manual start lets us run each workflow step in a separate notebook cell.

In [ ]:
profile_context = profile("Beginner RAG Workflow")
workflow = profile_context.__enter__()
print("Started:", workflow.name)

## 8. Step 1: User Prompt

In [ ]:
with step("User Prompt", type="planner", prompt=user_question):
    pass

print("User prompt recorded")

## 9. Step 2: Retrieve Chunks

In [ ]:
with step("Retrieve Policy Chunks", type="retriever", chunk_count=5, avg_tokens_per_chunk=30) as s:
    start = time.time()
    chunks = retrieve_context(user_question, top_k=5)
    s.step.metrics.latency = time.time() - start
    s.step.metadata["retrieved_chunks"] = chunks
    s.step.metadata["chunk_count"] = len(chunks)

for i, chunk in enumerate(chunks, start=1):
    print(f"{i}. {chunk}")

## 10. Step 3: Generate Answer

In [ ]:
with step("Generate Answer", type="llm_call", provider="openai", model=MODEL_NAME) as s:
    start = time.time()
    response = call_llm(user_question, chunks)
    s.record(response)
    s.step.metrics.latency = time.time() - start

answer = response.choices[0].message.content
print(answer)

## 11. Close Workflow

In [ ]:
profile_context.__exit__(None, None, None)
print("Total tokens:", workflow.total_tokens)
print("Total cost:", workflow.total_cost)

## 12. View Steps as a Table

In [ ]:
def workflow_to_dataframe(workflow):
    rows = []
    for st in workflow.steps:
        rows.append({
            "step": st.name,
            "type": st.type.value if hasattr(st.type, "value") else str(st.type),
            "prompt_tokens": st.metrics.prompt_tokens or 0,
            "completion_tokens": st.metrics.completion_tokens or 0,
            "total_tokens": st.metrics.total_tokens or 0,
            "cost_usd": st.metrics.cost or 0,
            "latency_sec": st.metrics.latency or 0,
        })
    return pd.DataFrame(rows)

rag_df = workflow_to_dataframe(workflow)
rag_df

## 13. Visualize Tokens by Step

In [ ]:
plt.figure(figsize=(9, 4))
plt.bar(rag_df["step"], rag_df["total_tokens"])
plt.title("RAG Workflow: Tokens by Step")
plt.ylabel("Total Tokens")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

## 14. Visualize Latency by Step

In [ ]:
plt.figure(figsize=(9, 4))
plt.bar(rag_df["step"], rag_df["latency_sec"])
plt.title("RAG Workflow: Latency by Step")
plt.ylabel("Latency Seconds")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

## 15. Save, Report, and Analyze

In [ ]:
JSONExporter().export(workflow, "beginner_rag_report.json")
print("Saved beginner_rag_report.json")

In [ ]:
!agenticlens report beginner_rag_report.json

In [ ]:
!agenticlens analyze beginner_rag_report.json

## What This Notebook Proves

AgenticLens can track a RAG workflow and show token usage, latency, cost, and optimization suggestions.